In [ ]:
!pip install xgboost lightgbm optuna scikit-learn pandas numpy -q
print("All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.6 MB/s eta 0:00:00
All libraries installed!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Dataset download karo
import os
if not os.path.exists('Boom-Challenge-Datasets'):
    os.system('git clone https://github.com/poweredbyfreelancer/Boom-Challenge-Datasets')

# Data load karo
train = pd.read_csv('Boom-Challenge-Datasets/forward_prediction/train.csv')
labels = pd.read_csv('Boom-Challenge-Datasets/forward_prediction/train_labels.csv')
test = pd.read_csv('Boom-Challenge-Datasets/forward_prediction/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Labels shape:", labels.shape)
print("\nData loaded successfully!")

Train shape: (2930, 8)
Test shape: (492, 8)
Labels shape: (2930, 6)

Data loaded successfully!


In [ ]:
def create_features(df):
    df = df.copy()

    # Physics-based features
    df['energy_per_gravity'] = df['energy'] / (df['gravity'] + 0.001)
    df['strength_porosity'] = df['strength'] * (1 - df['porosity'])
    df['impact_factor'] = df['energy'] * df['shape_factor'] * np.sin(df['angle_rad'])
    df['density_factor'] = df['strength'] / (df['porosity'] + 0.001)
    df['coupling_energy'] = df['coupling'] * df['energy']
    df['gravity_angle'] = df['gravity'] * np.cos(df['angle_rad'])
    df['atmosphere_strength'] = df['atmosphere'] * df['strength']
    df['total_energy'] = df['energy'] * df['coupling'] * df['shape_factor']

    return df

train_fe = create_features(train)
test_fe = create_features(test)

print("Original features:", train.shape[1])
print("New features:", train_fe.shape[1])
print("Feature names:", train_fe.columns.tolist())

Original features: 8
New features: 16
Feature names: ['porosity', 'atmosphere', 'gravity', 'coupling', 'strength', 'shape_factor', 'energy', 'angle_rad', 'energy_per_gravity', 'strength_porosity', 'impact_factor', 'density_factor', 'coupling_energy', 'gravity_angle', 'atmosphere_strength', 'total_energy']


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
import lightgbm as lgb

# Data prepare
X = train_fe.values
y = labels.values
X_test = test_fe.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# Model 1 - XGBoost
print("Training XGBoost...")
xgb_model = MultiOutputRegressor(
    xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
)
xgb_model.fit(X_scaled, y)
print("XGBoost done!")

# Model 2 - LightGBM
print("Training LightGBM...")
lgb_model = MultiOutputRegressor(
    lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        random_state=42,
        verbose=-1
    )
)
lgb_model.fit(X_scaled, y)
print("LightGBM done!")

# Model 3 - Random Forest
print("Training Random Forest...")
rf_model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        random_state=42
    )
)
rf_model.fit(X_scaled, y)
print("Random Forest done!")

print("\nAll 3 models trained!")

Training XGBoost...
XGBoost done!
Training LightGBM...
LightGBM done!
Training Random Forest...
Random Forest done!

All 3 models trained!


In [ ]:
# Teeno models ki predictions combine karo
xgb_pred = xgb_model.predict(X_test_scaled)
lgb_pred = lgb_model.predict(X_test_scaled)
rf_pred = rf_model.predict(X_test_scaled)

# Weighted average — XGBoost aur LightGBM ko zyada weight
final_pred = (xgb_pred * 0.4) + (lgb_pred * 0.4) + (rf_pred * 0.2)

# Submission file banao
submission = pd.DataFrame(final_pred, columns=labels.columns)
submission.insert(0, 'id', range(1, len(submission)+1))
submission.to_csv('pro_submission.csv', index=False)

print("Pro submission ready!")
print(submission.head())

Pro submission ready!
   id         P80  fines_frac  oversize_frac         R95   R50_fines  \
0   1  132.847975    0.136592       0.370126  824.651341  785.760553   
1   2  114.211092    0.044153       0.173595  220.824865  237.202482   
2   3  134.674892    0.180230       0.362963  851.895677  804.450442   
3   4  157.682231    0.006400       0.473400  484.432254  579.177244   
4   5  118.123515    0.056034       0.205405  247.386150  286.163475   

   R50_oversize  
0    366.568153  
1    112.813899  
2    394.333061  
3    262.649999  
4    111.440598  


In [ ]:
from scipy.optimize import differential_evolution

# Smart search using better ranges from training data
np.random.seed(42)
n_scenarios = 500000

scenarios = pd.DataFrame({
    'porosity': np.random.uniform(0.0, 0.35, n_scenarios),
    'atmosphere': np.random.uniform(0.05, 0.85, n_scenarios),
    'gravity': np.random.uniform(3.7, 9.81, n_scenarios),
    'coupling': np.random.uniform(0.4, 1.6, n_scenarios),
    'strength': np.random.uniform(0.8, 3.8, n_scenarios),
    'shape_factor': np.random.uniform(0.75, 1.35, n_scenarios),
    'energy': np.random.uniform(2.6, 4.6, n_scenarios),
    'angle_rad': np.random.uniform(0.52, 1.31, n_scenarios)
})

# Feature engineering apply karo
scenarios_fe = create_features(scenarios)
X_scen = scaler.transform(scenarios_fe.values)

# Ensemble predict karo
xgb_s = xgb_model.predict(X_scen)
lgb_s = lgb_model.predict(X_scen)
rf_s = rf_model.predict(X_scen)
pred_scen = (xgb_s * 0.4) + (lgb_s * 0.4) + (rf_s * 0.2)

pred_df = pd.DataFrame(pred_scen, columns=labels.columns)

# Filter constraints
mask = (pred_df['P80'] >= 96) & (pred_df['P80'] <= 101) & (pred_df['R95'] <= 175)
valid = scenarios[mask].copy()

print("Valid scenarios found:", len(valid))

if len(valid) >= 20:
    # Lowest energy wale 20 select karo — small impact score maximize hoga
    valid_with_pred = valid.copy()
    valid_with_pred['R95_pred'] = pred_df['R95'][mask].values
    valid_sorted = valid_with_pred.sort_values('energy').head(20)

    final_inverse = valid_sorted.drop('R95_pred', axis=1).reset_index(drop=True)
    final_inverse.insert(0, 'id', range(1, 21))
    final_inverse.to_csv('pro_inverse_design.csv', index=False)
    print("Pro inverse design ready!")
    print(final_inverse[['id','energy','gravity','P80_pred'] if 'P80_pred' in final_inverse.columns else ['id','energy','gravity']].head())
else:
    print("Kam scenarios mile — adjust kar rahe hain...")

Valid scenarios found: 8142
Pro inverse design ready!
   id    energy   gravity
0   1  2.600089  8.825099
1   2  2.600376  7.095876
2   3  2.600475  5.597503
3   4  2.600888  5.995314
4   5  2.601133  7.078696


In [ ]:
from google.colab import files
files.download('pro_submission.csv')
files.download('pro_inverse_design.csv')
print("Both files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Both files downloaded!
